# Module 8: Capstone: Decision-Memo System

Combine all four multi-agent patterns into one complete pipeline: from a decision brief to an approved leadership memo.

![Decision-Memo Capstone: Orchestrator (P5 Agent-as-Tool) delegates to Researcher (P1), Risk Analyzer and Customer Analyzer running in parallel (P2 Fork-Join), then Writer+Critic quality loop (P3 Critic-Refiner), producing the Leadership Memo](./architecture.png)

**Patterns combined:**
- **P1 Sequential**: Researcher gathers data first, result passed forward
- **P2 Fork-Join**: Risk Analyzer and Customer Analyzer run in parallel with `asyncio.gather`
- **P3 Critic-Refiner**: quality gate: Writer drafts, Critic checks, Revisor refines until APPROVED
- **P5 Agent-as-Tool**: Orchestrator delegates to the three sub-pipelines as callable tools

**Sub-agents in this pipeline:** Planner, Researcher, Risk Analyzer, Customer Analyzer, Synthesizer, Critic, Revisor

**Prerequisites:** All previous modules (1-7). This module reuses tools from Module 2.

## Components in This Module

| Component | Pattern | What it does |
|-----------|---------|-------------|
| `researcher_agent` | P1 Sequential | Gathers company data, benchmarks, competitor intel: runs first |
| `parallel_analyzers` | P2 Fork-Join | Runs 3 option analyzers simultaneously via `asyncio.gather` |
| `critic_refiner` | P3 Critic-Refiner | Writer drafts memo → Critic approves or requests revision → loop |
| `orchestrator` | P5 Agent-as-Tool | Coordinates all three specialists; LLM decides the call sequence |

In [ ]:
!uv pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster, lower cost):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
print("✅ Setup complete!")

In [ ]:
import sys, os, time, asyncio, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

import nest_asyncio
nest_asyncio.apply()

from strands import Agent, tool
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

---

## Part 1: System Prompts

All four specialists share one model instance (passed at Agent creation). Narrow, focused prompts: each agent does exactly one job.

In [ ]:
RESEARCHER_PROMPT = (
    "You are a market research specialist. Use your tools to gather relevant data. "
    "Return structured findings. data only, no recommendations."
)

ANALYZER_PROMPT = (
    "You are a business strategy analyst. Evaluate the ONE option you are given: "
    "strengths, weaknesses, complexity (Low/Med/High), top 2 risks with mitigations, verdict. "
    "100 words max."
)

WRITER_PROMPT = (
    "You are an executive memo writer. Write a COMPLETE leadership memo with:\n"
    "## Recommendation (one sentence: which option and why)\n"
    "## Options at a Glance (table comparing A, B, C)\n"
    "## Top 3 Risks with specific mitigations\n"
    "## Success Metrics (at least 2 KPIs with numeric targets)\n"
    "## Decision Required (owner, deadline, who approves)\n"
    "If you receive feedback, revise and include ALL sections."
)

CRITIC_PROMPT = (
    "You are a quality critic. Check ONLY these 5 criteria:\n"
    "1. ## Recommendation section with a clear option choice\n"
    "2. ## Options at a Glance table comparing A, B, C\n"
    "3. ## Top 3 Risks with at least 3 risks each with a mitigation\n"
    "4. ## Success Metrics with at least 2 KPIs with numeric targets\n"
    "5. ## Decision Required with owner AND deadline\n"
    "Respond: APPROVED or REVISION NEEDED: [criteria numbers missing]"
)

ORCHESTRATOR_PROMPT = (
    "You are a strategic decision analyst coordinating the Decision Intelligence pipeline.\n"
    "Steps:\n"
    "1. Call researcher_agent to gather market data for the brief.\n"
    "2. Call parallel_analyzers with the brief and research findings. runs A/B/C simultaneously.\n"
    "3. Call critic_refiner with the brief and combined analyses, produces the quality-checked memo.\n"
    "Execute all three steps in order. Do not skip any step."
)

---

## Part 2: Build the Three Specialist Tools

Each tool wraps a complete sub-pipeline. The orchestrator sees them as single callable functions.

In [ ]:
@tool
def researcher_agent(topic: str) -> str:
    '''Research market context, company data, benchmarks, and competitive intelligence for a decision topic.

    Args:
        topic: The decision topic or brief to research
    '''
    worker = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCHER_PROMPT,
        callback_handler=None,
    )
    return str(worker(topic))

In [ ]:
@tool
def parallel_analyzers(brief: str, research_context: str) -> str:
    '''Run all three option analyzers (A, B, C) in parallel and return their combined assessments.
    Use AFTER researcher_agent. Do not call this before research is complete.

    Args:
        brief: The original decision brief
        research_context: Research findings from researcher_agent
    '''
    a = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
    b = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
    c = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)

    async def fork():
        return await asyncio.gather(
            a.invoke_async(f"Option A: Exclusive Premium ($19.99/mo, invite-only top 10%)\nBrief: {brief}\nResearch: {research_context}"),
            b.invoke_async(f"Option B: Gradual Rollout ($14.99/mo, 5% A/B pilot)\nBrief: {brief}\nResearch: {research_context}"),
            c.invoke_async(f"Option C: Full Launch ($12.99/mo, open to all, 30-day trial)\nBrief: {brief}\nResearch: {research_context}"),
        )

    ra, rb, rc = asyncio.run(fork())
    return f"OPTION A:\n{ra}\n\nOPTION B:\n{rb}\n\nOPTION C:\n{rc}" 

In [ ]:
@tool
def critic_refiner(brief: str, analyses: str) -> str:
    '''Draft and quality-check the executive memo through a critic feedback loop.
    Use AFTER parallel_analyzers. Returns the final approved memo.

    Args:
        brief: The original decision brief
        analyses: Combined analyses of all three options from parallel_analyzers
    '''
    writer = Agent(name="writer", system_prompt=WRITER_PROMPT, callback_handler=None)
    critic = Agent(name="critic", system_prompt=CRITIC_PROMPT, callback_handler=None)

    def needs_revision(state):
        r = state.results.get("critic")
        return bool(r) and "revision needed" in str(r.result).lower()

    builder = GraphBuilder()
    builder.add_node(writer, "writer")
    builder.add_node(critic, "critic")
    builder.set_entry_point("writer")
    builder.add_edge("writer", "critic")
    builder.add_edge("critic", "writer", condition=needs_revision)
    builder.set_max_node_executions(6)
    builder.set_execution_timeout(120)
    builder.reset_on_revisit(True)

    graph = builder.build()
    result = graph(f"Brief:\n{brief}\n\nOption analyses:\n{analyses}")

    for node in reversed(result.execution_order):
        if node.node_id == "writer":
            return str(node.result)
    return str(result)

---

## Part 3: The Orchestrator and Full Brief

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Options:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

orchestrator = Agent(
    tools=[researcher_agent, parallel_analyzers, critic_refiner],
    system_prompt=ORCHESTRATOR_PROMPT,
)

print("Running full Decision Intelligence pipeline...")
t0 = time.time()
result = orchestrator(DECISION_BRIEF)
elapsed = time.time() - t0
print(f"\n⏱️  Pipeline completed in {elapsed:.1f}s")

---

## Part 4: Inspect the Pipeline

In [ ]:
# Show which tools were called and in what order
print("=== PIPELINE EXECUTION ===")
call_n = 0
for msg in orchestrator.messages:
    for block in msg.get("content", []):
        if "toolUse" in block:
            call_n += 1
            tu = block["toolUse"]
            inp_keys = list(tu.get("input", {}).keys())
            print(f"  {call_n}. {tu['name']}({', '.join(inp_keys)})")

print()
print("Pattern 5 (orchestrator) called Pattern 1, then Pattern 2, then Pattern 3.")
print("The LLM decided the order: no Python routing code was written.")

In [ ]:
# Token usage across the full pipeline
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

print(f"{'Metric':<25} {'Value':>10}")
print("-" * 37)
print(f"{'Input tokens':<25} {usage.get('inputTokens', 0):>10,}")
print(f"{'Output tokens':<25} {usage.get('outputTokens', 0):>10,}")
print(f"{'Total tokens':<25} {usage.get('totalTokens', 0):>10,}")
print(f"{'Orchestrator cycles':<25} {summary.get('total_cycles', 'n/a'):>10}")
print()

tool_stats = summary.get("tool_usage", {})
if tool_stats:
    print("Per-tool timing:")
    for name, data in tool_stats.items():
        s = data.get("execution_stats", {})
        print(f"  {name}: calls={s.get('call_count',0)} | avg_time={round(s.get('average_time',0),1)}s")

---

## What You Built

A production-grade Decision Intelligence System that combines four multi-agent patterns:

| Pattern | Component | Strands API |
|---------|-----------|-------------|
| Sequential (P1) | Research phase: first, then analyze, then write | Python sequence |
| Fork-Join (P2) | Three option analyzers run simultaneously | `asyncio.gather` + `invoke_async` |
| Critic-Refiner (P3) | Quality gate on the final memo | `GraphBuilder` + cycle edge |
| Agent-as-Tool (P5) | Orchestrator delegates to all three | `@tool` wrapping `Agent` |

---

## Key Takeaways

The complete system from input brief to approved leadership memo in ~50 seconds:
- 1 orchestrator (P5) coordinating 3 specialist tools
- 1 researcher calling 3 business intelligence tools (P1)
- 3 parallel option analyzers (P2)
- 1 writer + 1 critic in a quality loop (P3)

**Next:** Module 8 deploys this system to Amazon Bedrock AgentCore Runtime: managed compute, scalable endpoints, production observability.